# CHIRPS–ET–Groundwater Analysis Script

This script processes CHIRPS precipitation data, extracts site-level precipitation using spatial buffers, and combines it with ET and USGS groundwater data to compute ET:precipitation ratios and groundwater decline trends across study sites.

In [10]:
# Import libraries
import os
import requests
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import rasterio
from scipy.stats import spearmanr
import glob
from rasterio.mask import mask
from shapely.geometry import Point, box
from pyproj import Transformer

In [11]:
# Read in ET and groundwater data for all sites
et_gw_merged_all_sites = "/capstone/aridgw/outputs/2km/gwl_cult_et_2km.csv"
et_gw_merged_all_sites = pd.read_csv(et_gw_merged_all_sites)
et_gw_merged_all_sites

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg
0,2000,KSGS.371852100505801,239.390000,72.966072,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,724.822
1,2001,KSGS.371852100505801,241.960000,73.749408,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,767.646
2,2002,KSGS.371852100505801,242.780000,73.999344,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,768.023
3,2003,KSGS.371852100505801,246.710000,75.197208,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,790.507
4,2004,KSGS.371852100505801,247.710000,75.502008,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,743.859
...,...,...,...,...,...,...,...,...,...,...,...
986,2014,willcox,346.899946,105.735100,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,805.900
987,2015,willcox,352.000011,107.289600,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,709.303
988,2017,willcox,372.750012,113.614200,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,601.719
989,2019,willcox,392.100078,119.512100,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,656.576


## Creating a precipitation normal raster using 2000–2020 CHIRPS data to match the temporal extent of the study dataset.

In [ ]:
# Takes 23 mins to run

# Define folder containing daily CHIRPS raster files
data_dir = "/capstone/aridgw/data/chirps_daily_data/"

# Select only CHIRPS raster files from 2000–2020
files = sorted(glob.glob(data_dir + "*.tif"))
files = [f for f in files if any(str(year) in f for year in range(2000, 2021))]
print(len(files), "files found")

# Read first file to get metadata
with rasterio.open(files[0]) as src:
    meta = src.meta.copy()
    shape = src.shape

# Create array to store summed precipitation values
precip_sum = np.zeros(shape, dtype=np.float64)

# Loop through rasters and sum precipitation values
for i, f in enumerate(files):
    with rasterio.open(f) as src:
        precip_sum += src.read(1)
    if i % 100 == 0:
        print(f"Processed {i}/{len(files)}")

# Calculate mean daily precipitation raster
precip_mean = precip_sum / len(files)

# Save precipitation normal raster to a remote server
meta.update(dtype=rasterio.float32)
with rasterio.open("/capstone/aridgw/data/chirps_daily_precip_normal_2000_2020.tif", 'w', **meta) as dst:
    dst.write(precip_mean.astype(np.float32), 1)

# Save precipitation normal raster locally to `outputs` folder
#meta.update(dtype=rasterio.float32)
#with rasterio.open("../outputs/chirps_daily_precip_normal_2000_2020.tif", 'w', **meta) as dst:
#    dst.write(precip_mean.astype(np.float32), 1)

7671 files found
Processed 0/7671
Processed 100/7671
Processed 200/7671
Processed 300/7671
Processed 400/7671
Processed 500/7671
Processed 600/7671
Processed 700/7671
Processed 800/7671
Processed 900/7671
Processed 1000/7671
Processed 1100/7671
Processed 1200/7671
Processed 1300/7671
Processed 1400/7671
Processed 1500/7671
Processed 1600/7671
Processed 1700/7671
Processed 1800/7671
Processed 1900/7671
Processed 2000/7671
Processed 2100/7671
Processed 2200/7671
Processed 2300/7671
Processed 2400/7671
Processed 2500/7671
Processed 2600/7671
Processed 2700/7671
Processed 2800/7671
Processed 2900/7671
Processed 3000/7671
Processed 3100/7671
Processed 3200/7671
Processed 3300/7671
Processed 3400/7671
Processed 3500/7671
Processed 3600/7671
Processed 3700/7671
Processed 3800/7671
Processed 3900/7671
Processed 4000/7671
Processed 4100/7671
Processed 4200/7671
Processed 4300/7671
Processed 4400/7671
Processed 4500/7671
Processed 4600/7671
Processed 4700/7671
Processed 4800/7671
Processed 4900/

##  The mean is calculated by averaging all raster pixel values that intersect the buffered region, with each intersecting pixel contributing equally regardless of how much of its area overlaps the buffer.

## To change the buffer size, edit `buffer_km=0.5` to half the length of the buffer you want (example: for a 10 km buffer use `buffer_km=5`) in the following line:
`def extract_precip_with_buffer(lon, lat, src, buffer_km=0.5):`

## When changing buffer size, you have to also update all related file names and labels, including:
-   Output filenames (e.g., 1km → 2km, 4km, 10km)
-   Variable names that include resolution or scale (e.g., et_precip_ratio_1km)
-   Folder names if used (e.g., /outputs/1km/)

The computation code (masking, CRS transform, mean extraction) does not need to change—only the buffer parameter and naming conventions.

In [12]:
et_gw_merged_all_sites = et_gw_merged_all_sites.copy()

# Define function to extract precipitation values around each site
# Change `buffer_km=0.5` ⚠️
def extract_precip_with_buffer(lon, lat, src, buffer_km=1):

    # Transform coordinates to raster coordinate system
    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    x, y = transformer.transform(lon, lat)

    point = Point(x, y)

    # Create square buffer around site
    half_size = buffer_km / 111.32
    geom = box(
        x - half_size, y - half_size,
        x + half_size, y + half_size
    )
    
    # Mask raster to buffered site area
    out_image, out_transform = mask(
        src,
        [geom],
        crop=True,
        all_touched=True
    )

    data = out_image[0]

    # Return mean precipitation value for buffered area
    return data.mean() if data.size > 0 else np.nan

# Extract precipitation values for all sites
# Change `buffer_km=0.5` ⚠️
with rasterio.open("/capstone/aridgw/outputs/chirps_daily_precip_normal_2000_2020.tif") as src:
    precip_values = [
        extract_precip_with_buffer(lon, lat, src, buffer_km=1)
        for lon, lat in zip(
            et_gw_merged_all_sites["longitude"],
            et_gw_merged_all_sites["latitude"]
        )
    ]

# Add precipitation normals to dataframe
et_gw_merged_all_sites["chirps_precip_normal"] = precip_values
et_gw_merged_all_sites

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg,chirps_precip_normal
0,2000,KSGS.371852100505801,239.390000,72.966072,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,724.822,1.511885
1,2001,KSGS.371852100505801,241.960000,73.749408,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,767.646,1.511885
2,2002,KSGS.371852100505801,242.780000,73.999344,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,768.023,1.511885
3,2003,KSGS.371852100505801,246.710000,75.197208,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,790.507,1.511885
4,2004,KSGS.371852100505801,247.710000,75.502008,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,743.859,1.511885
...,...,...,...,...,...,...,...,...,...,...,...,...
986,2014,willcox,346.899946,105.735100,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,805.900,0.912725
987,2015,willcox,352.000011,107.289600,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,709.303,0.912725
988,2017,willcox,372.750012,113.614200,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,601.719,0.912725
989,2019,willcox,392.100078,119.512100,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,656.576,0.912725


In [13]:
# Create copy of dataframe for groundwater trend calculations
df = et_gw_merged_all_sites.copy()

# Define function to calculate groundwater trends
def calc_trend(group):
    # Remove rows with missing groundwater values
    g = group.dropna(subset=["year_value", "depth_to_gw_m"])
    if len(g) < 2:
        return np.nan
    
    # Calculate groundwater trend using linear regression slope (m/year)
    slope = np.polyfit(g["year_value"], g["depth_to_gw_m"], 1)[0]
    return slope

# Calculate groundwater trends for each site
gw_trends = df.groupby("site_id").apply(calc_trend).reset_index()
gw_trends.columns = ["site_id", "gw_trend_m_per_yr"]

# Calculate climate variables for each site
climate = df.groupby("site_id").agg(
    region=("region", "first"),
    mean_et=("scaled_annual_et_avg", "mean"),
    mean_precip=("chirps_precip_normal", "mean")
).reset_index()

# Multiply by 365 to get from mm/day to mm/year
climate["mean_precip"] = climate["mean_precip"] * 365

# Calculate ET to precipitation ratio
climate["et_precip_ratio"] = climate["mean_et"] / climate["mean_precip"]

# Merge groundwater trends with climate variables
site_summary = gw_trends.merge(climate, on="site_id")
site_summary

/tmp/ipykernel_1215206/1897879128.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gw_trends = df.groupby("site_id").apply(calc_trend).reset_index()


,site_id,gw_trend_m_per_yr,region,mean_et,mean_precip,et_precip_ratio
0,KSGS.371852100505801,0.616701,Southern Kansas,777.084238,551.838196,1.408174
1,KSGS.372043101363101,0.213301,Southern Kansas,539.564857,511.470032,1.054930
2,KSGS.372539100142504,1.047972,Southern Kansas,825.893095,605.336914,1.364353
3,KSGS.373331098033301,0.063359,Southern Kansas,842.846000,928.159485,0.908083
4,KSGS.373607100565301,0.595615,Southern Kansas,628.959429,523.986206,1.200336
5,KSGS.374111099070401,0.176172,Southern Kansas,783.701810,737.784912,1.062236
6,KSGS.374125100344101,1.690761,Southern Kansas,729.846333,576.212463,1.266627
7,KSGS.374747100552101,1.912739,Southern Kansas,867.576571,526.280457,1.648506
8,KSGS.375145100485701,1.217462,Southern Kansas,877.892810,531.417358,1.651984
9,KSGS.375454101075401,1.541198,Southern Kansas,925.517381,532.699219,1.737411


In [14]:
# Save site_summary as a csv to outputs folder in remote server
site_summary.to_csv("/capstone/aridgw/outputs/2km/site_summary_2km.csv", index=False)

# Save site_summary as a csv to outputs folder locally
site_summary.to_csv("../outputs/site_summary_2km.csv", index=False)

In [15]:
# Merge summary variables back into full dataset
et_gw_merged_all_sites = et_gw_merged_all_sites.merge(
    site_summary[['site_id', 'mean_et', 'mean_precip', 'et_precip_ratio']],
    on = 'site_id',
    how = 'left'
)

# Remove temporary precipitation normal column
et_gw_merged_all_sites = et_gw_merged_all_sites.drop(columns="chirps_precip_normal")
et_gw_merged_all_sites

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg,mean_et,mean_precip,et_precip_ratio
0,2000,KSGS.371852100505801,239.390000,72.966072,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,724.822,777.084238,551.838196,1.408174
1,2001,KSGS.371852100505801,241.960000,73.749408,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,767.646,777.084238,551.838196,1.408174
2,2002,KSGS.371852100505801,242.780000,73.999344,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,768.023,777.084238,551.838196,1.408174
3,2003,KSGS.371852100505801,246.710000,75.197208,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,790.507,777.084238,551.838196,1.408174
4,2004,KSGS.371852100505801,247.710000,75.502008,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,743.859,777.084238,551.838196,1.408174
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986,2014,willcox,346.899946,105.735100,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,805.900,746.277333,333.144501,2.240101
987,2015,willcox,352.000011,107.289600,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,709.303,746.277333,333.144501,2.240101
988,2017,willcox,372.750012,113.614200,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,601.719,746.277333,333.144501,2.240101
989,2019,willcox,392.100078,119.512100,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,656.576,746.277333,333.144501,2.240101


In [16]:
# Save et_gw_merged_all_sites as a csv to outputs folder in remote server
et_gw_merged_all_sites.to_csv(
    "/capstone/aridgw/outputs/2km/et_precipt_ratio_2km.csv",
    index = False
)

# Save et_gw_merged_all_sites as a csv to outputs folder locally
et_gw_merged_all_sites.to_csv(
    "../outputs/et_precipt_ratio_2km.csv",
    index = False
)

In [17]:
# Read in final ET to precipitation ratio dataset
et_gw_merged_all_sites = "/capstone/aridgw/outputs/2km/et_precipt_ratio_2km.csv"
et_gw_merged_all_sites = pd.read_csv(et_gw_merged_all_sites)
et_gw_merged_all_sites

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg,mean_et,mean_precip,et_precip_ratio
0,2000,KSGS.371852100505801,239.390000,72.966072,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,724.822,777.084238,551.8382,1.408174
1,2001,KSGS.371852100505801,241.960000,73.749408,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,767.646,777.084238,551.8382,1.408174
2,2002,KSGS.371852100505801,242.780000,73.999344,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,768.023,777.084238,551.8382,1.408174
3,2003,KSGS.371852100505801,246.710000,75.197208,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,790.507,777.084238,551.8382,1.408174
4,2004,KSGS.371852100505801,247.710000,75.502008,37.31502,-100.8505,USGS,Southern Kansas,2,2.0,743.859,777.084238,551.8382,1.408174
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986,2014,willcox,346.899946,105.735100,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,805.900,746.277333,333.1445,2.240101
987,2015,willcox,352.000011,107.289600,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,709.303,746.277333,333.1445,2.240101
988,2017,willcox,372.750012,113.614200,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,601.719,746.277333,333.1445,2.240101
989,2019,willcox,392.100078,119.512100,32.03619,-109.7540,Jasechko,Southwest US,2,2.0,656.576,746.277333,333.1445,2.240101
